# Income Growth Across Global Deciles

This notebook follows the project's World Bank PIP distribution and global-decile approach. It contains two related views:

- **Annual growth over time:** year on the x-axis, annual percent change in mean welfare on the y-axis, one line per global decile.
- **Elephant-style summary:** global deciles on the x-axis, cumulative percent growth in mean welfare over a selected period on the y-axis.

The PIP file contains population-weighted country income bins, not person or household IDs, so growth is calculated from each year's global-decile mean, not by following the same people over time. Deciles are recalculated each year to match the rest of the project.

## Import Libraries

In [ ]:
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

DATA_FILE = Path('GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.csv')
FIG_DIR = Path('figures/draft')
FIRST_YEAR = 1990
LAST_YEAR = 2025  # 2026 is a forecast in this PIP release.
RECENT_BASE_YEAR = 2015
FIG_DIR.mkdir(parents=True, exist_ok=True)
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Load and Review the Data

Each row is a country-year welfare bin with the population it represents. `welf` is daily welfare in 2021 PPP dollars; `pop` is population in millions. There is no person-level panel identifier. The missingness and type checks below make that structure explicit before calculating growth.

In [2]:
distribution = pd.read_csv(
    DATA_FILE,
    usecols=['year', 'code', 'welf', 'pop'],
)
distribution = distribution[distribution['year'].between(FIRST_YEAR, LAST_YEAR)].copy()

print(f'Rows: {len(distribution):,}')
print(f'Countries: {distribution["code"].nunique():,}')
print(f'Years: {distribution["year"].min()}-{distribution["year"].max()}')
display(distribution.dtypes.to_frame('dtype'))
display(distribution[['year', 'code', 'welf', 'pop']].isna().sum().to_frame('missing values'))

assert distribution['year'].between(FIRST_YEAR, LAST_YEAR).all()
assert (distribution['pop'].dropna() > 0).all()
assert (distribution['welf'].dropna() >= 0).all()

Rows: 7,848,000
Countries: 218
Years: 1990-2025


,dtype
year,int64
code,str
welf,float64
pop,float64


,missing values
year,0
code,0
welf,0
pop,0


## Assign Global Deciles and Calculate Growth

For each year, pool country bins, sort them by welfare, and split represented population fractionally at the exact 10% global-population boundaries. Deciles are recalculated each year to match the main exercise. Annual growth is the percent change in a decile's mean welfare from the previous year. Since the PIP file has no person or household IDs, this is growth of a rank group's mean, not a longitudinal change for fixed people. Zero or missing previous means are excluded from percentage-growth calculations.

In [3]:
def global_decile_means(year_bins: pd.DataFrame) -> pd.DataFrame:
    """Return exact population-weighted welfare means for global deciles."""
    bins = year_bins.dropna(subset=['welf', 'pop']).sort_values('welf', kind='stable')
    welfare = bins['welf'].to_numpy()
    population = bins['pop'].to_numpy()
    cumulative_end = population.cumsum()
    cumulative_start = cumulative_end - population
    total_population = population.sum()

    rows = []
    for decile in range(1, 11):
        lower = (decile - 1) * total_population / 10
        upper = decile * total_population / 10
        overlap = np.maximum(
            0,
            np.minimum(cumulative_end, upper) - np.maximum(cumulative_start, lower),
        )
        decile_population = overlap.sum()
        rows.append({
            'year': int(bins['year'].iloc[0]),
            'decile': decile,
            'population_m': decile_population,
            'mean_welfare': np.average(welfare, weights=overlap),
        })

    return pd.DataFrame(rows)


decile_means = pd.concat(
    [global_decile_means(year_bins) for _, year_bins in distribution.groupby('year')],
    ignore_index=True,
).sort_values(['decile', 'year'])

decile_means['previous_mean'] = decile_means.groupby('decile')['mean_welfare'].shift(1)
decile_means['annual_growth_pct'] = (
    100 * (decile_means['mean_welfare'] / decile_means['previous_mean'] - 1)
)
decile_means.loc[decile_means['previous_mean'].le(0), 'annual_growth_pct'] = np.nan
decile_means['decile_label'] = 'D' + decile_means['decile'].astype(str)

decile_population_check = decile_means.groupby('year')['population_m'].agg(['min', 'max'])
display(decile_population_check.round(3).head())
assert (decile_means['mean_welfare'] > 0).all()
assert (decile_population_check['max'] - decile_population_check['min'] < 1e-8).all()

,min,max
year,,
1990,529.925,529.925
1991,538.850,538.850
1992,547.712,547.712
1993,556.478,556.478
1994,565.085,565.085


## Summarize Growth by Year and Decile

The table below keeps the decile-year observations and the population represented in each decile. A separate period summary compares each decile's mean welfare in 1990, 2015, and 2025.

In [4]:
mean_welfare_wide = decile_means.pivot(index='decile', columns='year', values='mean_welfare')
period_summary = pd.DataFrame({
    'decile': mean_welfare_wide.index,
    'mean_welfare_1990': mean_welfare_wide[FIRST_YEAR].to_numpy(),
    'mean_welfare_2015': mean_welfare_wide[RECENT_BASE_YEAR].to_numpy(),
    'mean_welfare_2025': mean_welfare_wide[LAST_YEAR].to_numpy(),
})
period_summary['growth_1990_2025_pct'] = (
    100 * (period_summary['mean_welfare_2025'] / period_summary['mean_welfare_1990'] - 1)
)
period_summary['growth_2015_2025_pct'] = (
    100 * (period_summary['mean_welfare_2025'] / period_summary['mean_welfare_2015'] - 1)
)
period_summary['decile_label'] = 'D' + period_summary['decile'].astype(str)

display(period_summary.round(2))

reported_d1_growth = period_summary.loc[period_summary['decile'].eq(1), 'growth_2015_2025_pct'].item()
print(f'D1 mean-welfare growth, 2015-2025: {reported_d1_growth:.1f}%')
assert round(reported_d1_growth, 1) == 4.1

,decile,mean_welfare_1990,mean_welfare_2015,mean_welfare_2025,growth_1990_2025_pct,growth_2015_2025_pct,decile_label
0,1,0.89,1.88,1.96,118.94,4.14,D1
1,2,1.46,3.14,3.81,161.30,21.04,D2
2,3,1.93,4.08,5.18,168.68,26.85,D3
3,4,2.46,5.16,6.53,165.22,26.41,D4
4,5,3.13,6.62,8.29,164.77,25.30,D5
5,6,4.16,8.73,10.96,163.57,25.50,D6
6,7,6.28,12.08,15.35,144.31,27.12,D7
7,8,11.73,18.23,23.25,98.10,27.50,D8
8,9,25.51,32.90,40.21,57.62,22.21,D9
9,10,70.94,88.76,103.48,45.86,16.58,D10


D1 mean-welfare growth, 2015-2025: 4.1%


In [5]:
decile_order = [f'D{i}' for i in range(1, 11)]
decile_colors = [
    '#34A7F2', '#666666', '#777777', '#888888', '#999999',
    '#A3A3A3', '#ADADAD', '#B7B7B7', '#C1C1C1', '#CBCBCB',
]

annual_growth_data = decile_means.dropna(subset=['annual_growth_pct'])
annual_growth_lines = alt.Chart(annual_growth_data).mark_line(strokeWidth=1.8).encode(
    x=alt.X('year:Q', title='Year', scale=alt.Scale(domain=[FIRST_YEAR + 1, LAST_YEAR], nice=False),
            axis=alt.Axis(format='d', values=list(range(1995, LAST_YEAR + 1, 5)))),
    y=alt.Y('annual_growth_pct:Q', title='Annual growth in mean welfare (%)'),
    color=alt.Color('decile_label:N', sort=decile_order,
                    scale=alt.Scale(domain=decile_order, range=decile_colors),
                    legend=alt.Legend(title='Global decile', orient='bottom', columns=5)),
)
annual_zero = alt.Chart(pd.DataFrame({'growth': [0]})).mark_rule(color='#888888', strokeWidth=0.8).encode(
    y='growth:Q',
)
annual_growth_chart = (annual_zero + annual_growth_lines).properties(
    width=650,
    height=340,
    title=alt.TitleParams(
        'Annual income growth by global decile',
        subtitle='Percent change in mean daily welfare from the previous year; deciles are recalculated each year',
    ),
)
annual_growth_chart

alt.LayerChart(...)

## Elephant-Style Summary by Decile

Here the x-axis is the global decile, not the year. Each point shows the cumulative percent change in mean welfare over the stated period. The 1990-2025 curve summarizes the full series; the 2015-2025 curve focuses on the recent period. Because those windows have different lengths, compare the shape within each curve rather than treating their cumulative growth levels as directly comparable.

In [9]:
elephant_data = pd.concat([
    period_summary.assign(period='1990-2025', growth_pct=period_summary['growth_1990_2025_pct']),
    period_summary.assign(period='2015-2025', growth_pct=period_summary['growth_2015_2025_pct']),
], ignore_index=True)
elephant_data['zero'] = 0

curve_x = alt.X('decile_label:N', sort=decile_order, title='Global income decile (D1 = poorest)')
curve_y = alt.Y('growth_pct:Q', title='Cumulative growth in mean welfare (%)')
curve_line = alt.Chart().mark_line(color='#9E9E9E', strokeWidth=2.2).encode(
    x=curve_x,
    y=curve_y,
)
curve_points = alt.Chart().mark_point(size=55, filled=True).encode(
    x=curve_x,
    y=curve_y,
    color=alt.condition(alt.datum.decile == 1, alt.value('#34A7F2'), alt.value('#9E9E9E')),
    tooltip=[
        alt.Tooltip('period:N', title='Period'),
        alt.Tooltip('decile_label:N', title='Decile'),
        alt.Tooltip('growth_pct:Q', title='Growth (%)', format='.1f'),
    ],
)
curve_zero = alt.Chart().mark_rule(color='#888888', strokeWidth=0.8).encode(
    y='zero:Q',
)
elephant_chart = alt.layer(curve_zero, curve_line, curve_points, data=elephant_data).properties(width=430, height=240).facet(
    column=alt.Facet('period:N', sort=['1990-2025', '2015-2025'], title=None,
                     header=alt.Header(labelFontSize=12, labelFontWeight='bold')),
).properties(
    title=alt.TitleParams(
        'How mean income growth varied across global deciles',
        subtitle='Elephant-style view at decile resolution; PIP welfare in 2021 PPP dollars per person per day',
    ),
)

growth_chart_path = FIG_DIR / 'annual_growth_by_decile.png'
elephant_chart_path = FIG_DIR / 'elephant_style_growth_by_decile.png'
annual_growth_chart.save(str(growth_chart_path), scale_factor=2)
elephant_chart.save(str(elephant_chart_path), scale_factor=2)
elephant_chart

alt.FacetChart(...)